In [ ]:
from simtk.openmm.app import *
from simtk.openmm import *
from simtk.unit import *

import mdtraj as md
import nglview as nv
import PeptideBuilder.Geometry
from PeptideBuilder import Geometry
import os
import ipykernel
import ipywidgets
import pdbfixer


In [ ]:
seqlist = 'GGG','GPG','GAG','GVG','GLG','GIG','GMG','GCG','GFG','GYG','GWG','GHG','GKG','GRG','GQG','GNG','GEG','GDG','GSG','GTG'

path = '/Users/williambrown/Box/GROUP_data/Will/gxg-project/trajectories/15-each/0.05mM/'


In [ ]:
###                                                     ###
### My attempt at generalizing the code for any peptide ###
###                                                     ###

def pepbuild(seq):
    
    res1 = Geometry.geometry(seq[0])
    
    structure = PeptideBuilder.initialize_res(res1)
    
    for letter in seq[1:]:
        new_res = Geometry.geometry(letter)
        PeptideBuilder.add_residue(structure, new_res)
        
    PeptideBuilder.add_terminal_OXT(structure)

    import Bio.PDB
    out = Bio.PDB.PDBIO()
    out.set_structure(structure)
    out.save(path + seq + '.pdb')
    
    return structure
    

In [ ]:
for i in seqlist:
    pepbuild(i)
        

In [ ]:
###                                                      ###
### Sometimes we need to fix PDBs for unnatural residues ###
###                                                      ###

from pdbfixer import PDBFixer

#RNA
#fixer = PDBFixer(filename='ru40.pdb')
#fixer.findMissingResidues()
#fixer.findMissingAtoms()
#fixer.addMissingAtoms()
#fixer.addMissingHydrogens(7.0)
#PDBFile.writeFile(fixer.topology, fixer.positions, open('ru40_2.pdb', 'w'))

def pepfix(pdb):
    fixer = PDBFixer(filename = pdb)
    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.removeHeterogens(True)
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.0)
    PDBFile.writeFile(fixer.topology, fixer.positions, open(pdb, 'w'))

###                                                               ###
### MANUALLY GO IN AND FIX FIRST FEW LINES UNTIL O5 IS FIRST ATOM ###
###                                                               ###

In [ ]:
pdbs = 'GGG.pdb','GPG.pdb','GAG.pdb','GVG.pdb','GLG.pdb','GIG.pdb','GMG.pdb','GCG.pdb','GFG.pdb','GYG.pdb','GWG.pdb','GHG.pdb','GKG.pdb','GRG.pdb','GQG.pdb','GNG.pdb','GEG.pdb','GDG.pdb','GSG.pdb','GTG.pdb'

In [ ]:
for i in pdbs:
    pepfix(path + i)

In [ ]:
!cat mixture.inp #packmol input script

!packmol < mixture.inp

In [ ]:
###                                                   ###
### THIS CELL CAN BE RUN IF FORCEFIELD HAS .XML FILES ###
###                                                   ###


#Make sure O5-P3 terminal residues are recognized, delete a few lines before or after if needed.

pdb        = PDBFile('packed.pdb')

modeller   = Modeller(pdb.topology, pdb.positions)
modeller.addHydrogens()

forcefield = ForceField('amber14-all.xml', 'amber14/tip3p.xml')
modeller.addSolvent(forcefield, model='tip3p',  boxSize=Vec3(10.0, 10.0, 10.0)*nanometers, ionicStrength=0.1*molar)

system = forcefield.createSystem(modeller.topology, nonbondedMethod=PME)
integrator = VerletIntegrator(0.001*picoseconds)
simulation = Simulation(modeller.topology, system, integrator)
simulation.context.setPositions(modeller.positions)
simulation.minimizeEnergy(maxIterations=100)

print('Saving PDB...')
positions = simulation.context.getState(getPositions=True).getPositions()
PDBFile.writeFile(simulation.topology, positions, open('output.pdb', 'w'))


In [ ]:
###                                                    ###
### NEEDS TO BE DONE FOR FORCEFIELDS WITH NO .XML FILE ###
###                                                    ###

### Need to be run in terminal to generate gro and top files + itps ###

!gmx pdb2gmx -f output.pdb -o output.gro -p output.top -ignh


In [ ]:
from simtk.openmm.app import *
from simtk.openmm import *
from simtk.unit import *
from sys import stdout

gro = GromacsGroFile('output.gro')
top = GromacsTopFile('output.top', unitCellDimensions=gro.getUnitCellDimensions(), includeDir='/usr/local/gromacs/share/gromacs/top')
system = top.createSystem(nonbondedMethod=PME, nonbondedCutoff=1*nanometer,
        constraints=HBonds)
integrator = LangevinIntegrator(300*kelvin, 1/picosecond, 0.002*picoseconds)
simulation = Simulation(top.topology, system, integrator)
simulation.context.setPositions(gro.positions)
simulation.minimizeEnergy()
simulation.reporters.append(PDBReporter('output_2.pdb', 1000))
simulation.reporters.append(StateDataReporter(stdout, 1000, step=True,
        potentialEnergy=True, temperature=True))
simulation.step(10000)

In [ ]:
#Input structures from X3DNA, peptide builder, Pymol, etc. 
traj = md.load_pdb('output_2.pdb')
view = nv.show_mdtraj(traj)

#view.clear_representations()
view.add_representation('ball+stick', selection='protein')
view

In [ ]:
from Bio.SeqUtils import molecular_weight
import numpy as np 
import math

In [ ]:
density = 100 
l = 10
mass1 = molecular_weight('GRG', "protein")
mass2 = molecular_weight('GIG', "protein")
totmass = (mass1 + mass2) / 2
totmass


In [ ]:
nchains = density * l**3/ totmass
nchains

In [ ]:
ratio = (2,1)
ratio_sum = ratio[0]+ratio[1]
ratio_sum

In [ ]:
n1 = nchains / ratio_sum * ratio[0]
n2 = nchains / ratio_sum * ratio[1]

print(n1)
print(n2)


In [ ]:
mass = (n1*mass1)+(n2*mass2)#)/6.022e23/1000
ml = l**3

mgml = mass/ml
mgml

In [ ]:
seq_num_list = [('GTG', 1),('GCG',5),('GIG',3)]

In [ ]:
from Bio.SeqUtils import molecular_weight

ratio_list = []
seq_list = []
for i in seq_num_list:
    ratio_list.append(i[1])
    seq_list.append(i[0])
ratio_tot = sum(ratio_list)

keys = []
values = []
for i in seq_list:
    keys.append(i)
    values.append(molecular_weight(i, "protein"))
    mass_dict = { key : value for key, value in zip(keys, values)}

sum(list(mass_dict.values()))


In [ ]:
def num_chains(d, l, seq_num_list):
    '''d = density (mg/mL)
       l = box size (angstroms)
       seq_num_list = [(seq1, #),(seq2, #),(seq3, #)] #List of tuples
       ratio = [seq1] / [seq2]
       calculates and returns number of 
       peptide/protein chains needed to pack
    '''
    from Bio.SeqUtils import molecular_weight
    import numpy as np 
    import math

    ratio_list = []
    seq_list = []
    for i in seq_num_list:
        ratio_list.append(i[1])
        seq_list.append(i[0])
    ratio_tot = sum(ratio_list)

    keys = []
    values = []
    for i in seq_list:
        keys.append(i)
        values.append(molecular_weight(i, "protein"))
        mass_dict = {key : value for key, value in zip(keys, values)}

    mass_list = list(mass_dict.values())
    #mult_list = []
    #for num1, num2 in zip(mass_list, ratio_list):
    #    mult_list.append(num1*num2)
    avg_mass = sum(mass_list)/len(seq_list)

    l = l/10 #convert to nm
    nchains = d * (l**3 * 1e-21) / (avg_mass/6.022e23*1000)
    
    ratio_list = []
    for i in seq_num_list:
        ratio_list.append(i[1])

    ratio_tot = sum(ratio_list)

    keys2 = []
    values2 = []
    for i in seq_num_list:
        keys2.append(i[0])
        values2.append(nchains / ratio_tot * i[1])
        chain_dict = {key : value for key, value in zip(keys2, values2)}
        
    return chain_dict

In [ ]:
seq_num_list = [('GWG', 3), ('GIG', 1), ('GYG', 5)]
num_chains(125, 100, seq_num_list)

In [ ]:
def run_packmol(d, l, seq_num_list, fname, nchains):

    ratio_list = []
    seq_list = []
    for i in seq_num_list:
        ratio_list.append(i[1])
        seq_list.append(i[0])
    
    make_pdbs(seq_list)

    for i in pdb_list
        from pdbfixer import PDBFixer
        fixer = PDBFixer(filename = i)
        fixer.findMissingResidues()
        fixer.findMissingAtoms()
        fixer.addMissingAtoms()
        fixer.addMissingHydrogens(7.0)
        PDBFile.writeFile(fixer.topology, fixer.positions, open(i, 'w'))

    ### prep packmol file
    packmol_str = f'''
        #
        # {header}
        #
        tolerance 2.0
        filetype pdb
        output {fname + '_packed.pdb'}
  
        #add TER to after each monomer chain
        add_amber_ter
 
        for i in pdb_list:
            structure {i}
            for item in chain_dict:
                if i[:2]==chain_dict[item]:
                    number {chain_dict.values(chain_dict[item])}
            inside box 0. 0. 0. {l} {l} {l}
            end structure'''
    
    packmol_file = open(f"{fname}_mixture.inp", 'w')
    packmol_file.write(packmol_str)
    packmol_file.close()

In [ ]:
def make_pdbs(seq_num_list):
    
    '''make .pdb files for peptide sequences (1 Letter seqs)'''

    import PeptideBuilder.Geometry
    from PeptideBuilder import Geometry

    ratio_list = []
    seq_list = []
    for i in seq_num_list:
        ratio_list.append(i[1])
        seq_list.append(i[0])

    pdb_list = []

    for i in seq_list:
        res1 = Geometry.geometry(i[0])

        structure = PeptideBuilder.initialize_res(res1)

        for letter in i[1:]:
            new_res = Geometry.geometry(letter)
            PeptideBuilder.add_residue(structure, new_res)

        PeptideBuilder.add_terminal_OXT(structure)

        import Bio.PDB
        out = Bio.PDB.PDBIO()
        out.set_structure(structure)
        out.save(i + '.pdb')

        pdb_list.append(i + '.pdb')

    for i in pdb_list:
        from pdbfixer import PDBFixer
        from openmm.app import PDBFile
        fixer = PDBFixer(filename = i)
        fixer.findMissingResidues()
        fixer.findMissingAtoms()
        fixer.addMissingAtoms()
        fixer.addMissingHydrogens(7.0)
        PDBFile.writeFile(fixer.topology, fixer.positions, open(i, 'w'))

    return pdb_list

In [ ]:
seq_num_list = [('GWG', 3), ('GIG', 1), ('GYG', 5)]

pdb_list = make_pdbs(seq_num_list)

chain_dict = num_chains(125, 100, seq_num_list)
pdb_list

In [ ]:
chain_dict

In [ ]:
header = 'GWG,GIG,GYG'
fname = 'WIY' 

    
packmol_str = f
    #
    # {header}
    #
    tolerance 2.0
    filetype pdb
    output {fname+'_packed.pdb'}
  
    #add TER to after each monomer chain
    add_amber_ter
 
    for i in pdb_list:
        structure {i}
        for item in chain_dict:
            if i[:2]==chain_dict[item]:
                number {chain_dict.values(chain_dict[item])}
        inside box 0. 0. 0. {l} {l} {l}
        end structure


In [ ]:

def test_func(fname, header='A mixture of peptides', l=100, pdb_list=pdb_list, chain_dict = chain_dict):
        
        fname = ''
        for i in pdb_list:
            fname += i[1]

        str_a = f'''
#
# {header}
#

tolerance 2.0
filetype pdb
output {fname + '_packed.pdb'}
  
#add TER to after each monomer chain
add_amber_ter
        
'''

        for i in pdb_list:
            for key in chain_dict.keys():
                if key == i[:3]:
                    nchains = int(chain_dict[key])
                    str = f'''
structure {i}
number {nchains}
inside box 0. 0. 0. {l} {l} {l}
end structure
                                
            ''' 
                    str_a += str

        packmol_file = open(f"{fname}_mixture.inp", 'w')
        packmol_file.write(str_a)
        packmol_file.close()



In [ ]:
test_func(fname = 'charged')

In [ ]:
str_a = f'''
    #
    # 'header'
    #
    tolerance 2.0
    filetype pdb
    output 'name' + '_packed.pdb'

    #add TER to after each monomer chain
    add_amber_ter
        
    '''

for i in pdb_list:
    for key in chain_dict.keys():
        if key == i[:2]:
            nchains = chain_dict[key]
            str = '''structure {i}
                     number {nchains}
                     inside box 0. 0. 0. {l} {l} {l}
                     end structure
                                
                     ''' 
            str_a += str

            print(str_a)


In [ ]:
str_a

In [ ]:
def make_pdbs(seq_num_list):
    
    '''make .pdb files for peptide sequences (1 Letter seqs)'''

    import PeptideBuilder.Geometry
    from PeptideBuilder import Geometry

    ratio_list = []
    seq_list = []
    for i in seq_num_list:
        ratio_list.append(i[1])
        seq_list.append(i[0])

    pdb_list = []

    for i in seq_list:
        res1 = Geometry.geometry(i[0])

        structure = PeptideBuilder.initialize_res(res1)

        for letter in i[1:]:
            new_res = Geometry.geometry(letter)
            PeptideBuilder.add_residue(structure, new_res)

        PeptideBuilder.add_terminal_OXT(structure)

        import Bio.PDB
        out = Bio.PDB.PDBIO()
        out.set_structure(structure)
        out.save(i + '.pdb')

        pdb_list.append(i + '.pdb')

    for i in pdb_list:
        from pdbfixer import PDBFixer
        from openmm.app import PDBFile
        fixer = PDBFixer(filename = i)
        fixer.findMissingResidues()
        fixer.findMissingAtoms()
        fixer.addMissingAtoms()
        fixer.addMissingHydrogens(7.0)
        PDBFile.writeFile(fixer.topology, fixer.positions, open(i, 'w'))

    return pdb_list




def num_chains(d, l, seq_num_list):
    '''d = density (mg/mL)
       l = box size (angstroms)
       seq_num_list = [(seq1, #),(seq2, #),(seq3, #)] #List of tuples
       ratio = [seq1] / [seq2]
       calculates and returns number of 
       peptide/protein chains needed to pack
    '''
    from Bio.SeqUtils import molecular_weight
    import numpy as np 
    import math

    ratio_list = []
    seq_list = []
    for i in seq_num_list:
        ratio_list.append(i[1])
        seq_list.append(i[0])
    ratio_tot = sum(ratio_list)

    keys = []
    values = []
    for i in seq_list:
        keys.append(i)
        values.append(molecular_weight(i, "protein"))
        mass_dict = {key : value for key, value in zip(keys, values)}

    mass_list = list(mass_dict.values())
    avg_mass = sum(mass_list)/len(seq_list)

    l = l/10 #convert to nm
    nchains = d * (l**3 * 1e-21) / (avg_mass/6.022e23*1000)
    
    ratio_list = []
    for i in seq_num_list:
        ratio_list.append(i[1])

    ratio_tot = sum(ratio_list)

    keys2 = []
    values2 = []
    for i in seq_num_list:
        keys2.append(i[0])
        values2.append(nchains / ratio_tot * i[1])
        chain_dict = {key : value for key, value in zip(keys2, values2)}
        
    return chain_dict





def run_packmol(seq_num_list, fname = 'RLM', d=125, l=100, header='A mixture of peptides'):

        ratio_list = []
        seq_list = []
        fname_list = []
        for i in seq_num_list:
            ratio_list.append(i[1])
            seq_list.append(i[0])

        pdb_list = make_pdbs(seq_num_list)

        for i in seq_list:
            fname_list.append(i[1])
            fname = ''.join(fname_list)

        chain_dict = num_chains(d, l, seq_num_list)

        ### prep packmol file
        str_a = f'''
#
# {header}
#

tolerance 2.0
filetype pdb
output {fname + '_packed.pdb'}
  
#add TER to after each monomer chain
add_amber_ter
        
'''

        for i in pdb_list:
            for key in chain_dict.keys():
                if key == i[:3]:
                    nchains = int(chain_dict[key])
                    str = f'''
structure {i}
number {nchains}
inside box 0. 0. 0. {l} {l} {l}
end structure
                                
            ''' 
                    str_a += str

        packmol_file = open(f"{fname}_mixture.inp", 'w')
        packmol_file.write(str_a)
        packmol_file.close()



    


In [ ]:
seq_num_list = [('GRG', 5), ('GLG', 3), ('GMG', 1)]
run_packmol(seq_num_list)

In [ ]:
seq_list = run_packmol(seq_num_list)

make_pdbs(seq_list)